# Overlap and divergence of anomaly scores

In [4]:
import os
# Set environment variables to disable multithreading
# as users will probably want to set the number of cores
# to the max of their computer.
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from anomaly.constants import GALAXY_LINES
from anomaly.utils import specobjid_to_idx
from anomaly.utils import VelocityFilter
from anomaly.utils import AnomalyOverlapAnalyzer
from autoencoders.ae import AutoEncoder

from sdss.metadata import MetaData

meta = MetaData()

# Constants

# Custom functions

## IDs top anomalies

In [6]:
def get_ids_set(score, df, quantile=99, n_top=None, use_ntop=False):

    if use_ntop is False:
        
        quantile *= 0.01
        thresh = df[score].quantile(quantile)
        ids = set(df[df[score] > thresh].index)
        
    else:

        ids = set(
            df[score].sort_values(
                ascending=False
            ).iloc[:n_top].index
        )

    return ids

## SDSS photometry

In [7]:
def get_sdss_spec_img(
    specobjid, ra, dec, save_to
    ):
    """Download sdss image and spectrum"""

    meta.download_sdss_spectrum_image(
        specobjid=specobjid,
        save_to=save_to,
        image_format="jpeg",
    )

    meta.get_sdss_image(
        specobjid,
        coordinates= (ra, dec),
        save_to=save_to,
        image_format="jpeg",
    )

## thumbnails

In [8]:
def spec_photo_thumbnail(
    ax,
    wave, spec,
    photo_img=None,
    width_height_image='100%',
    bbox_to_anchor=(0.695, 0.52, 0.5, 0.5),
    add_image=False,
):

    ax.plot(wave, spec, color='black')
    # Create inset in top-right corner of figure

    ax.set_xlim(350, 770)
    wave_ticks = list(np.arange(400, 751, 50))
    ax.set_xticks(wave_ticks)

    if add_image is False:
        return ax, None

    axins = inset_axes(
        ax,
        width=width_height_image, height=width_height_image,
        bbox_to_anchor=bbox_to_anchor,
        bbox_transform=ax.transAxes,
    )

    # Show image in the inset
    axins.imshow(photo_img)
    axins.axis('off')  # Hide axis around the image

    return ax, axins


## Line indicators

In [9]:
GALAXY_LINES_NM = {
    'OII': {'name': 'OII', 'line_wave': 372.7},
    # 'NeIII': {'name': 'NeIII', 'line_wave': 386.9},
    # 'HI': {'name': 'HI', 'line_wave': 388.9},
    'H_epsilon': {'name': r'H$\epsilon$', 'line_wave': 397},
    'H_delta': {'name': r'H$\delta$', 'line_wave': 410.2},
    'H_gamma': {'name': r'H$\gamma$', 'line_wave': 434.0},
    'H_beta': {'name': r'H$\beta$', 'line_wave': 486.1},
    'OIII': {'name': 'OIII', 'line_wave': 500.7},
    'H_alpha': {'name': r'H$\alpha$', 'line_wave': 656.3},
    'SII': {'name': 'SII', 'line_wave': 673},
    # 'SII': 671.6
}

In [10]:
def add_line_indicators(
    ax, wave, spec,
    indicator_starts,
    indicator_height,
    add_oii=False,
    add_ne3_he1=False,
    add_h_epsilon=False,
    add_h_delta=False,
    add_h_gamma=False,
    add_oiii=False,
    add_h_beta=False,
    add_sii=False,
    delta=1,
    indicator_color='blue',
    indicator_label_color='black',
    lw=1.5,
    fontsize=8,
):
    
    for line_name, value_dict in GALAXY_LINES_NM.items():
        
        # skip desired lines
        if line_name == 'OII' and add_oii is False:
            continue

        if line_name == 'H_epsilon' and add_h_epsilon is False:
            continue

        if line_name == 'H_delta' and add_h_delta is False:
            continue

        if line_name == 'H_gamma' and add_h_gamma is False:
            continue

        if line_name == 'H_beta' and add_h_beta is False:
            continue

        if line_name == 'OIII' and add_oiii is False:
            continue

        if line_name == 'SII' and add_sii is False:
            continue

        line_wave = value_dict['line_wave']
        # Find the flux value of the spectrum at the line's wavelength
        # get small neigborhood among line_wave
        # get max flux there and set as line_flux
        line_flux = np.max(
            spec[(wave > line_wave - delta) & (wave < line_wave + delta)]
        )

        # Plot vertical line indicator
        y_start_indicator = line_flux + indicator_starts
        y_end_indicator = y_start_indicator + indicator_height

        ax.plot(
            [line_wave, line_wave], 
            [y_start_indicator, y_end_indicator], 
            color=indicator_color, lw=lw
        )
        
        clean_name = value_dict['name']

        y_text_start = y_end_indicator + indicator_height/2

        ax.text(
            line_wave, 
            y_text_start, 
            clean_name,
            ha='center', va='bottom', 
            fontsize=fontsize, rotation=90, 
            color=indicator_label_color
        )

        # doubles manual setup
        if line_name == 'OIII':
            ax.plot(
                [495.9, 495.9], 
                [y_start_indicator, y_end_indicator], 
                color=indicator_color, lw=lw
            )

        if line_name == 'H_alpha':

            y_nii_start = y_start_indicator - indicator_height/2
            y_nii_end = y_end_indicator - indicator_height/2

            # NII 1st
            ax.plot(
                [654.8, 654.8],
                [y_nii_start, y_nii_end],
                color=indicator_color, lw=lw
            )
            y_nii_text = y_nii_end + indicator_height/2
            ax.text(
                649, y_nii_text,
                'NII',
                ha='center', va='bottom', 
                fontsize=fontsize, rotation=90, 
                color=indicator_label_color
            )

            # NII 2nd
            ax.plot(
                [658.3, 658.3], 
                [y_nii_start, y_nii_end], 
                color=indicator_color, lw=lw
            )

            ax.text(
                664.1, y_nii_text,
                'NII',
                ha='center', va='bottom', 
                fontsize=fontsize, rotation=90, 
                color=indicator_label_color
            )

        if line_name == 'SII':
            ax.plot(
                [671.6, 671.6],
                [y_start_indicator, y_end_indicator], 
                color=indicator_color, lw=lw
            )

    # optionally add NeIII-1 and He I
    if add_ne3_he1 is True:

        line_flux = np.max(
            spec[(wave > 386.9 - delta) & (wave < 388.9 + delta)]
        )
        y_start_indicator = line_flux + indicator_starts
        y_end_indicator = y_start_indicator + indicator_height

        # add indicator for NeIII 386.9
        ax.plot(
            [386.9, 386.9], 
            [y_start_indicator, y_end_indicator], 
            color=indicator_color, lw=lw
        )
        # add indicator for HI 388.9
        ax.plot(
            [388.9, 388.9], 
            [y_start_indicator, y_end_indicator], 
            color=indicator_color, lw=lw
        )
        # add label NeIII + He I
        y_text_start = y_end_indicator + indicator_height/2

        ax.text(
            387.9, 
            y_text_start, 
            'NeIII + HeI',
            ha='center', va='bottom', 
            fontsize=fontsize, rotation=90, 
            color=indicator_label_color
        )

    return ax

# Config

## Directories

In [11]:
phd_dir = "/home/elom/phd"
thesis_dir = f"{phd_dir}/thesis"
ch4_dir = f"{thesis_dir}/chapters/04_figures"
data_dir = f"{phd_dir}/code"
spectra_dir = f"{data_dir}/spectra"
models_dir = f"{data_dir}/models"
latent_dir = f"{data_dir}/latent"
bins_ids = [f'bin_{i:02d}' for i in range(4)] 

## Data

In [12]:
wave = np.load(f"{spectra_dir}/wave_spectra_imputed.npy")
wave_nm = wave*0.1

spectra = np.load(
    f"{spectra_dir}/spectra_imputed.npy",
    mmap_mode="r"
)

final_meta_df = pd.read_csv(
    f"{spectra_dir}/final_spec_n_z_warning_drop.csv.gz",
    index_col="specobjid",
)

idx_id_spec = np.load(
    f"{spectra_dir}/ids_imputing.npy",
    mmap_mode='r'
)

## iforest scores

In [13]:
iforest_scores_df_dict = {}
# iforest_model_dict = {}
iforest_params = {
    'bin_00': 's256_e300_f100',
    'bin_01': 's256_e300_f75',
    'bin_02': 's256_e300_f100',
    'bin_03': 's128_e300_f100'
}

for bin_id in bins_ids:

    # load df of scores
    _df = pd.read_csv(
        f"{latent_dir}/{bin_id}/iforest/"
        f"iforest_scores_{bin_id}_{iforest_params[bin_id]}.csv",
        index_col='specobjid'
    )

    _df.columns = ['raw_iforest', 'iforest_score', 'rank_iforest']
    iforest_scores_df_dict[bin_id] = _df

In [14]:
bin_id = 'bin_03'
iforest_scores_df_dict[bin_id].sort_values(by='rank_iforest', ascending=True).head()

,raw_iforest,iforest_score,rank_iforest
specobjid,,,
969534273977608192,-0.172210,1.000000,1
1538002139001939968,-0.168068,0.988482,2
561847176611784704,-0.166951,0.985377,3
2829471472643237888,-0.157599,0.959367,4
2811439757043722240,-0.148941,0.935291,5


## LOF scores

In [15]:
lof_scores_df_dict = {}
# lof_model_dict = {}
n = 60
metric = 'manhattan'

for bin_id in bins_ids:

    # load df of scores
    _df = pd.read_csv(
        f"{latent_dir}/{bin_id}/lof/"
        f"lof_scores_{bin_id}_n{n}_{metric}.csv",
        index_col='specobjid'
    )

    _df.columns = ['lof', 'lof_score', 'rank_lof']
    lof_scores_df_dict[bin_id] = _df
    
    # # load models
    # load_from = os.path.join(
    #     latent_dir, bin_id,
    #     f'lof_{bin_id}_n{n}_{metric}.joblib'
    # )

    # model = joblib.load(load_from)
    # lof_model_dict[bin_id] = model

In [16]:
bin_id = 'bin_01'
lof_scores_df_dict[bin_id].sort_values(by='rank_lof', ascending=True).head()

,lof,lof_score,rank_lof
specobjid,,,
2034543258029287424,-4.606827,1.000000,1
650786678071388160,-4.522794,0.981759,2
1857818737302857728,-3.762106,0.816637,3
1099043276306016256,-3.408893,0.739965,4
1623712368703858688,-3.286496,0.713397,5


## Join scores

In [17]:
scores_df_dict = {}

for bin_id in bins_ids:

    _df = lof_scores_df_dict[bin_id].join(
        iforest_scores_df_dict[bin_id],
        how='inner'
    ).copy()

    scores_df_dict[bin_id] = _df


In [18]:
bin_id = 'bin_00'
scores_df_dict[bin_id].sort_values(by='rank_lof', ascending=True).head()

,lof,lof_score,rank_lof,raw_iforest,iforest_score,rank_iforest
specobjid,,,,,,
407585970297268224,-8.440069,1.000000,1,-0.119439,0.813470,342
2367797523948529664,-5.663374,0.671010,2,-0.180470,0.949756,39
2934148553238407168,-4.983568,0.590465,3,-0.177416,0.942936,49
2278872322632869888,-4.601294,0.545173,4,-0.179087,0.946667,42
1079775709385222144,-4.409762,0.522479,5,-0.182996,0.955396,31


# Pair wise overlap

In [19]:
for bin_id in bins_ids:

    score_df = scores_df_dict[bin_id]

    (
        ids_lof, ids_iforest,
        common_ids,
        only_in_lof_ids, only_in_iforest_ids
    ) = AnomalyOverlapAnalyzer.overlap_pair_scores(
        score_a='lof_score',
        score_b='iforest_score',
        df=score_df.copy(),
        quantile=99
    )

    print(bin_id) 
    ids_a_b = set(list(ids_lof) + list(ids_iforest))
    n_ids_a_b = len(ids_a_b)
    print(f"N in A or B: {n_ids_a_b}")
    # 
    n_common = len(common_ids)
    print(f"N common: {n_common}")
    #
    overlap_pct = (n_common/n_ids_a_b)*100
    print(f"Overlap: {overlap_pct:.2f}%")
    n_ids_only_lof = len(only_in_lof_ids)
    print(f"N only in A (B): {n_ids_only_lof}")
    print('-'*50)

bin_00
N in A or B: 3293
N common: 345
Overlap: 10.48%
N only in A (B): 1474
--------------------------------------------------
bin_01
N in A or B: 3236
N common: 402
Overlap: 12.42%
N only in A (B): 1417
--------------------------------------------------
bin_02
N in A or B: 3350
N common: 288
Overlap: 8.60%
N only in A (B): 1531
--------------------------------------------------
bin_03
N in A or B: 3120
N common: 518
Overlap: 16.60%
N only in A (B): 1301
--------------------------------------------------


# Common anomalies

## Cadidates selection

In [27]:
# selection of common anomalies
common_selected_ids = [
    # broad emission with strong OIII
    1633733043925575680,
    # broad emission lines with second OIII cut off in half
    465054141559891968,
    # featureless continuum with blue slope
    1119282266950887424,
    # evolved galaxy
    1444661293335209984,
    # emission line with strong OII and blue slope
    1959124163192973312,
    # emission line with bumpy continuum and red slope
    # has strong H alpha, OII and OII respectively
    2006491952928811008,
    # evolved galaxy with artifact around 540 nm
    2217029189603715072,
    # evolved galaxy with weak H alpha and SII
    # a data arfact around 370 nm, and a pronounced red slope
    761223822287333376,
    # broad emission with extrem strong OIII and h beta cut off in half
    1189119947666647040,
    # strong emission but data artifact: no OII and no OIII
    2930814289679771648
]
len(common_selected_ids)

10

In [28]:
bin_id = 'bin_03'
score_df = scores_df_dict[bin_id]

score_df.loc[common_selected_ids, ['rank_lof', 'rank_iforest']]

,rank_lof,rank_iforest
specobjid,,
1633733043925575680,7,93
465054141559891968,17,148
1119282266950887424,15,501
1444661293335209984,23,510
1959124163192973312,26,1443
2006491952928811008,27,1679
2217029189603715072,45,1348
761223822287333376,88,928
1189119947666647040,412,285


## Data prep

In [ ]:
overview_common_dict = {
    'blue': 1119282266950887424,
    'red': 761223822287333376, 
    'evolved': 1444661293335209984,
    'evolved_artifact': 2217029189603715072,
    'emission_blue_strong_OII': 1959124163192973312,
    'emission_red_bumpy': 2006491952928811008,
    '':,
    '': ,

}
# ----------------------------------------------------------
print('Get specs and load photometry')
# Get specs
spec_overview_common_dict = {}

save_to = f"{ch_4_dir}/{bin_id}/overview_common"
for key in overview_common_dict.keys():

    objid = overview_common_dict[key]
    spec_idx = specobjid_to_idx(
        objid, idx_id_spec
    )

    spec_overview_common_dict[key] = spectra[spec_idx]

    # # download image and spectrum
    # ra, dec = final_meta_df[['ra', 'dec']].loc[objid].values
    # get_sdss_spec_img(
    #     specobjid=objid, ra=ra, dec=dec, save_to=save_to
    # )

# Load photometry
img_overview_common_dict = {}

for key in overview_common_dict.keys():

    objid = overview_common_dict[key]
    img_name = f"image_{objid}.jpeg"
    img_ = mpimg.imread(f"{save_to}/{img_name}")
    img_overview_common_dict[key] = img_

## Figure

In [ ]:
fig, axs = plt.subplots(
    4, 2,
    sharex=True,
    figsize=(14, 7)
)
width_height_image = '100%'
# ------------------------------------------------------
# row 1
axs[0, 0], ax_img = spec_photo_thumbnail(
    axs[0,0],
    wave_nm, spec_overview_common_dict['narrow_line'],
    img_overview_common_dict['narrow_line'],
    width_height_image=width_height_image,
    add_image=True
)

axs[0, 0] = add_line_indicators(
    axs[0, 0], wave_nm, spec_overview_common_dict['narrow_line'],
    delta=1,
    indicator_starts=5,
    indicator_height=5,
    add_oii=True,
    add_ne3_he1=True,
    add_h_epsilon=True,
    add_h_delta=True,
    add_h_gamma=True,
    add_oiii=True,
    add_h_beta=True,
    add_sii=True,
    lw=1,
    fontsize=6,
)

axs[0, 0].set_ylim(-5, 115) 
flux_ticks = range(0, 101, 25)
axs[0,0].set_yticks(flux_ticks)

## 
axs[0, 1], ax_artifact = spec_photo_thumbnail(
    axs[0, 1],
    wave_nm, spec_overview_common_dict['broad_line_large_OIII'],
    img_overview_common_dict['broad_line_large_OIII'],
    width_height_image=width_height_image,
    add_image=True
)

axs[0, 1] = add_line_indicators(
    axs[0, 1], wave_nm, spec_overview_common_dict['broad_line_large_OIII'],
    delta=1,
    indicator_starts=3,
    indicator_height=3,
    add_ne3_he1=True,
    add_h_epsilon=True,
    add_h_delta=True,
    add_h_gamma=True,
    add_h_beta=True,
    add_sii=True,
    lw=1,
    fontsize=6,
)

axs[0, 1].set_ylim(-2, 61)
flux_ticks = range(0, 61, 20)
axs[0, 1].set_yticks(flux_ticks)

# ---------------------------------------------- 
# row 2
axs[1, 0], ax_img = spec_photo_thumbnail(
    axs[1, 0],
    wave_nm, spec_overview_common_dict['broad_emission_dips_OIII_half'],
    img_overview_common_dict['broad_emission_dips_OIII_half'],
    width_height_image=width_height_image,
    add_image=True
)
axs[1, 0] = add_line_indicators(
    axs[1, 0], wave_nm, spec_overview_common_dict['broad_emission_dips_OIII_half'],
    indicator_starts=0.5,
    indicator_height=0.5,
    add_oii=True,
    add_ne3_he1=True,
    add_h_beta=True,
    add_oiii=True,
    add_h_gamma=True,
    add_sii=True,
    lw=1,
    fontsize=6,
    delta=5
)

axs[1, 0].set_ylim(0, 10) 
flux_ticks = range(0, 10, 3)
axs[1,0].set_yticks(flux_ticks)

##
axs[1, 1], ax_star = spec_photo_thumbnail(
    axs[1, 1],
    wave_nm, spec_overview_common_dict['star_forming_step_blue_slope'],
    img_overview_common_dict['star_forming_step_blue_slope'],
    width_height_image=width_height_image,
    add_image=True
)

axs[1, 1] = add_line_indicators(
    axs[1, 1], wave_nm, spec_overview_common_dict['star_forming_step_blue_slope'],
    indicator_starts=2,
    indicator_height=2,
    add_oii=True,
    add_ne3_he1=True,
    add_h_beta=True,
    add_oiii=True,
    add_h_gamma=True,
    add_sii=True,
    lw=1,
    fontsize=6,
    # delta=10
)

axs[1, 1].set_ylim(0, 42) 
flux_ticks = range(0, 41, 10)
axs[1, 1].set_yticks(flux_ticks)

# ----------------------------------------------
# row 3
axs[2, 0], ax_img = spec_photo_thumbnail(
    axs[2, 0],
    wave_nm, spec_overview_common_dict['blue_bump_emission'],
    img_overview_common_dict['blue_bump_emission'],
    width_height_image=width_height_image,
    add_image=True
)
axs[2, 0] = add_line_indicators(
    axs[2, 0], wave_nm, spec_overview_common_dict['blue_bump_emission'],
    indicator_starts=0.4,
    indicator_height=0.4,
    add_oii=True,
    add_h_gamma=True,
    add_h_beta=True,
    add_oiii=True,
    add_sii=True,
    lw=1,
    fontsize=6,
)
axs[2, 0].set_ylim(0, 5.5) 
flux_ticks = range(0, 6, 1)
axs[2, 0].set_yticks(flux_ticks)

## 
axs[2, 1], ax_artifact = spec_photo_thumbnail(
    axs[2, 1],
    wave_nm, spec_overview_common_dict['passive_star'],
    img_overview_common_dict['passive_star'],
    width_height_image=width_height_image,
    bbox_to_anchor=(0.275, 0.07, 0.5, 0.5),
    add_image=True
)

axs[2, 1].set_ylim(0, 1.8) 
flux_ticks = [0, 0.5, 1, 1.5]
axs[2, 1].set_yticks(flux_ticks)

# ---------------------------------------------- 
# row 4
axs[3, 0], ax_img = spec_photo_thumbnail(
    axs[3, 0],
    wave_nm, spec_overview_common_dict['spike'],
    img_overview_common_dict['spike'],
    width_height_image=width_height_image,
    add_image=True
)

axs[3, 0].set_ylim(0, 55) 
flux_ticks = [0, 25, 50]
axs[3, 0].set_yticks(flux_ticks)

##
axs[3, 1], ax_artifact = spec_photo_thumbnail(
    axs[3, 1],
    wave_nm, spec_overview_common_dict['noise_forest'],
    img_overview_common_dict['noise_forest'],
    width_height_image=width_height_image,
    add_image=True,
    bbox_to_anchor=(-0.167, 0.52, 0.5, 0.5),
)

axs[3, 1].set_ylim(0, 4.2) 
flux_ticks = [0, 2, 4]
axs[3, 1].set_yticks(flux_ticks)

# ---------------------------------------------- 
axs[3, 0].set_xlabel(r"$\lambda$ [nm]")
axs[3, 1].set_xlabel(r"$\lambda$ [nm]")

for ax in axs.flatten(): ax.minorticks_on()
for i, ax in enumerate(axs.flatten()):
    if i%2==0:
        ax.set_ylabel("Flux")
# ---------------------------------
#  add letters to panels
for ax, letter in zip(axs.flatten()[:7], range(1, 8, 1)):
    ax.text(
        0.02, 0.9,
        f'({letter})',
        horizontalalignment="left", verticalalignment="center",
        transform=ax.transAxes,
        # fontsize="small",
    )
axs[3, 1].text(
    0.14, 0.9,
    '(8)',
    horizontalalignment="left", verticalalignment="center",
    transform=axs[3, 1].transAxes,
    # fontsize="small",
)
# ---------------------------------
plt.subplots_adjust(
    hspace=0.075,
    wspace=0.078
)

# fig.savefig(
#     f"{ch_4_dir}/{bin_id}/overview_common_anomalies.pdf",
#     bbox_inches='tight'
# )


# Disctinct anomalies